In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA silver;

--Create format table General_info
CREATE TABLE IF NOT EXISTS general_info
USING delta 
AS SELECT
    metadata.matchId as match_id,
    info.gameVersion as game_version,
    ROUND(info.gameDuration/60, 2) as game_duration_minutes
FROM workspace.bronze.b_matches
WHERE 1=0;

--Merge new data without remakes into General_info
MERGE INTO general_info AS g
USING (
    SELECT metadata.matchId AS match_id,
    info.gameVersion AS game_version,
    ROUND(info.gameDuration/60, 2) AS game_duration_minutes
    FROM workspace.bronze.b_matches
    WHERE info.gameDuration > (14*60)) AS b
ON g.match_id = b.match_id
WHEN NOT MATCHED THEN INSERT *;


In [0]:
%sql

--Create format table Player_info
CREATE TABLE IF NOT EXISTS player_info
USING delta
AS SELECT
    match_id,
    player_name.riotIdGameName AS player_name,
    player_name.individualPosition AS lane_position,
    player_name.championName AS champion,
    player_name.kills AS kills,
    player_name.deaths AS deaths,
    player_name.assists AS assists,
    player_name.win AS win,
    player_name.goldEarned AS gold_earned,
    player_name.totalMinionsKilled AS minions_killed,
    player_name.totalDamageDealtToChampions AS damage_dealt_to_champions,    
    player_name.visionScore AS vision_score
FROM (
    SELECT 
        metadata.matchId AS match_id,
        EXPLODE(info.participants) AS player_name
    FROM workspace.bronze.b_matches
    WHERE 1=0
);

--Merge new data without remakes into Player_info
MERGE INTO player_info AS p
USING (
    SELECT
    match_id,
    player_name.riotIdGameName AS player_name,
    player_name.individualPosition AS lane_position,
    player_name.championName AS champion,
    player_name.kills AS kills,
    player_name.deaths AS deaths,
    player_name.assists AS assists,
    player_name.win AS win,
    player_name.goldEarned AS gold_earned,
    player_name.totalMinionsKilled AS minions_killed,
    player_name.totalDamageDealtToChampions AS damage_dealt_to_champions,    
    player_name.visionScore AS vision_score
    FROM (
        SELECT
        metadata.matchId AS match_id,
        EXPLODE(info.participants) AS player_name
        FROM workspace.bronze.b_matches
        WHERE info.gameDuration > (14*60)
        )
    ) AS b
ON p.match_id = b.match_id
WHEN NOT MATCHED THEN INSERT *;